In [6]:
import pandas as pd
import sqlite3

In [7]:
#Ejercicio 1
conn = sqlite3.connect('clima_lab.db')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS registros (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ciudad TEXT NOT NULL,
    temp_c REAL,
    humedad INTEGER,
    precip_mm REAL,
    viento_kmh REAL,
    fecha TEXT
)
''')
conn.commit()

cursor.execute("SELECT name, sql FROM sqlite_master WHERE type='table' AND name='registros'")
tabla_info = cursor.fetchone()
print(f"Tabla '{tabla_info[0]}' creada.")
print(f"SQL Generado:\n{tabla_info[1]}\n")

cursor.execute("PRAGMA table_info(registros)")
columnas = cursor.fetchall()
print("Columnas:")
for col in columnas:
    print(f"- {col[1]} ({col[2]})")

conn.close()

Tabla 'registros' creada.
SQL Generado:
CREATE TABLE registros (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ciudad TEXT NOT NULL,
    temp_c REAL,
    humedad INTEGER,
    precip_mm REAL,
    viento_kmh REAL,
    fecha TEXT
)

Columnas:
- id (INTEGER)
- ciudad (TEXT)
- temp_c (REAL)
- humedad (INTEGER)
- precip_mm (REAL)
- viento_kmh (REAL)
- fecha (TEXT)


In [8]:
#Ejercicio 2
conn = sqlite3.connect('clima_lab.db')
cursor = conn.cursor()

registro_unico = ('Santiago', 18.5, 72, 1.2, 15.3, '2026-06-15')
cursor.execute('''
INSERT INTO registros (ciudad, temp_c, humedad, precip_mm, viento_kmh, fecha)
VALUES (?, ?, ?, ?, ?, ?)
''', registro_unico)

print(f"ID del nuevo registro insertado: {cursor.lastrowid}")

nuevos_registros = [
    ('Valparaíso', 15.2, 80, 3.5, 22.1, '2026-06-15'),
    ('Concepción', 12.8, 85, 8.0, 18.7, '2026-06-15'),
    ('Temuco', 9.3, 90, 12.5, 25.0, '2026-06-15'),
    ('La Serena', 17.8, 55, 0.0, 12.4, '2026-06-15')
]

cursor.executemany('''
INSERT INTO registros (ciudad, temp_c, humedad, precip_mm, viento_kmh, fecha)
VALUES (?, ?, ?, ?, ?, ?)
''', nuevos_registros)

conn.commit()

cursor.execute("SELECT COUNT(*) FROM registros")
total_registros = cursor.fetchone()[0]
print(f"Total de registros en la tabla: {total_registros}")

conn.close()

ID del nuevo registro insertado: 1
Total de registros en la tabla: 5


In [9]:
#Ejercicio 3

#Coneccion sqlite3
conn = sqlite3.connect('clima_lab.db')
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

print("--- Consulta 1: Listado completo ---")
cursor.execute("SELECT ciudad, temp_c, fecha FROM registros")
for row in cursor.fetchall():
    print(f"{row['ciudad']:<15} | {row['temp_c']}°C | {row['fecha']}")

print("\n--- Consulta 2: Ciudad más cálida ---")
cursor.execute("SELECT ciudad, temp_c FROM registros ORDER BY temp_c DESC")
ciudad_calida = cursor.fetchone()
print(f"La ciudad más cálida es {ciudad_calida['ciudad']} con {ciudad_calida['temp_c']}°C")

print("\n--- Consulta 3: Filtro < 13°C ---")
cursor.execute("SELECT ciudad, temp_c FROM registros WHERE temp_c < ?", (13.0,))
for row in cursor.fetchall():
    print(f"{row['ciudad']}: {row['temp_c']}°C")

print("\n--- Consulta 4: Estadísticas por ciudad ---")
cursor.execute('''
SELECT ciudad, AVG(temp_c) as prom_temp, MIN(humedad) as min_hum, MAX(humedad) as max_hum 
FROM registros 
GROUP BY ciudad 
ORDER BY prom_temp DESC
''')
for row in cursor.fetchall():
    print(f"{row['ciudad']:<15} | Promedio Temp: {row['prom_temp']:.2f}°C | Humedad Min: {row['min_hum']}% | Humedad Max: {row['max_hum']}%")

conn.close()

--- Consulta 1: Listado completo ---
Santiago        | 18.5°C | 2026-06-15
Valparaíso      | 15.2°C | 2026-06-15
Concepción      | 12.8°C | 2026-06-15
Temuco          | 9.3°C | 2026-06-15
La Serena       | 17.8°C | 2026-06-15

--- Consulta 2: Ciudad más cálida ---
La ciudad más cálida es Santiago con 18.5°C

--- Consulta 3: Filtro < 13°C ---
Concepción: 12.8°C
Temuco: 9.3°C

--- Consulta 4: Estadísticas por ciudad ---
Santiago        | Promedio Temp: 18.50°C | Humedad Min: 72% | Humedad Max: 72%
La Serena       | Promedio Temp: 17.80°C | Humedad Min: 55% | Humedad Max: 55%
Valparaíso      | Promedio Temp: 15.20°C | Humedad Min: 80% | Humedad Max: 80%
Concepción      | Promedio Temp: 12.80°C | Humedad Min: 85% | Humedad Max: 85%
Temuco          | Promedio Temp: 9.30°C | Humedad Min: 90% | Humedad Max: 90%


In [10]:
#Ejercicio 4
conn = sqlite3.connect('clima_lab.db')
cursor = conn.cursor()

try:
    print("--- Operación 1: UPDATE de Valparaíso ---")
    cursor.execute("UPDATE registros SET temp_c = ? WHERE ciudad = ?", (16.0, 'Valparaíso'))
    print(f"Filas modificadas: {cursor.rowcount}")
    
    cursor.execute("SELECT ciudad, temp_c FROM registros WHERE ciudad = 'Valparaíso'")
    print(f"Nuevo valor: {cursor.fetchall()}")

    print("\n--- Operación 2: UPDATE masivo (Humedad) ---")
    cursor.execute("UPDATE registros SET humedad = humedad + 5 WHERE temp_c < ?", (10.0,))
    print(f"Filas modificadas: {cursor.rowcount}")

    print("\n--- Operación 3: DELETE con condición ---")
    cursor.execute("DELETE FROM registros WHERE ciudad = ?", ('Temuco',))
    print(f"Filas eliminadas: {cursor.rowcount}")
    
    cursor.execute("SELECT COUNT(*) FROM registros")
    print(f"Registros restantes en total: {cursor.fetchone()[0]}")

    conn.commit()

except sqlite3.Error as e:
    print(f"Ocurrió un error en la base de datos: {e}")
    conn.rollback()
finally:
    conn.close()

--- Operación 1: UPDATE de Valparaíso ---
Filas modificadas: 1
Nuevo valor: [('Valparaíso', 16.0)]

--- Operación 2: UPDATE masivo (Humedad) ---
Filas modificadas: 1

--- Operación 3: DELETE con condición ---
Filas eliminadas: 1
Registros restantes en total: 4


In [11]:
#Ejercicio 5
df = pd.read_csv('S15_registros_climaticos.csv')
filas_originales = len(df)
print("--- Información del DataFrame Original ---")
df.info()
print("\n--- Total de Nulos ---")
print(df.isnull().sum())

df['ciudad'] = df['ciudad'].str.title()

df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce').dt.strftime('%Y-%m-%d')

df = df.drop_duplicates(subset=['ciudad', 'fecha'])

df = df[(df['temp_c'] >= -5) & (df['temp_c'] <= 50)]

filas_limpias = len(df)
filas_eliminadas = filas_originales - filas_limpias

conn = sqlite3.connect('clima_lab.db')
df.to_sql('registros_csv', conn, if_exists='replace', index=False)

print("\n" + "="*40)
print("--- Consultas SQL de Análisis ---")

q1 = '''
SELECT ciudad, AVG(temp_c) as temp_promedio 
FROM registros_csv 
GROUP BY ciudad 
ORDER BY temp_promedio DESC 
LIMIT 1
'''
df_q1 = pd.read_sql_query(q1, conn)
print(f"\n1. Ciudad con mayor temperatura promedio:\n{df_q1.to_string(index=False)}")

q2 = '''
SELECT ciudad, AVG(precip_mm) as prom_precip 
FROM registros_csv 
WHERE fecha BETWEEN '2026-01-01' AND '2026-03-31' 
GROUP BY ciudad
'''
df_q2 = pd.read_sql_query(q2, conn)
print(f"\n2. Promedio precipitación Ene-Mar 2026:\n{df_q2.to_string(index=False)}")

q3 = '''
SELECT ciudad, temp_c, fecha 
FROM registros_csv 
ORDER BY temp_c DESC 
LIMIT 5
'''
df_q3 = pd.read_sql_query(q3, conn)
print(f"\n3. Top 5 mayores temperaturas registradas:\n{df_q3.to_string(index=False)}")

print("\n" + "="*40)
print("--- Resumen Final ---")
print(f"Filas originales en CSV: {filas_originales}")
print(f"Filas eliminadas en limpieza: {filas_eliminadas}")
print(f"Total de filas cargadas a SQLite: {filas_limpias}")

conn.close()

--- Información del DataFrame Original ---
<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ciudad      208 non-null    str    
 1   temp_c      194 non-null    float64
 2   humedad     198 non-null    float64
 3   precip_mm   208 non-null    float64
 4   viento_kmh  208 non-null    float64
 5   fecha       208 non-null    str    
dtypes: float64(4), str(2)
memory usage: 13.7 KB

--- Total de Nulos ---
ciudad         0
temp_c        14
humedad       10
precip_mm      0
viento_kmh     0
fecha          0
dtype: int64

--- Consultas SQL de Análisis ---

1. Ciudad con mayor temperatura promedio:
     ciudad  temp_promedio
Antofagasta      20.673333

2. Promedio precipitación Ene-Mar 2026:
      ciudad  prom_precip
 Antofagasta     3.314286
  Concepción     3.075000
     Iquique     2.470000
   La Serena     3.514286
Puerto Montt     3.200000
    Rancagua     3.